# Projeções dos métodos — `m4_low`, semente 103

A figura apresenta as frentes completas não dominadas dos cinco métodos nas projeções $f_1 \times f_2$, $f_2 \times f_4$ e $f_3 \times f_1$. Todos os objetivos são normalizados pelos valores ideal e nadir verdadeiros do cenário. As estrelas rosas representam os quatro ótimos individuais verdadeiros.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from pymoo.util.nds.non_dominated_sorting import NonDominatedSorting

def project_root(start=Path.cwd()):
    path = start.resolve()
    for candidate in (path, *path.parents):
        if (candidate / 'configs' / 'full.json').exists():
            return candidate
    raise FileNotFoundError('Raiz do projeto não encontrada.')

ROOT = project_root()
SCENARIO = 'm4_low'
SEED = 103
ALPHA = 2 ** 0.75
METHOD_ORDER = ['CNBI', 'VRF-NBI', 'NSGA-III', 'MOEA/D', 'NBI']
METHOD_LABEL = {'CNBI': 'C-NBI', 'VRF-NBI': 'VRF-NBI', 'NSGA-III': 'NSGA-III', 'MOEA/D': 'MOEA/D', 'NBI': 'NBI original'}
METHOD_COLOR = {'CNBI': '#E69F00', 'VRF-NBI': '#0072B2', 'NSGA-III': '#009E73', 'MOEA/D': '#D62728', 'NBI': '#7B3294'}
STAR_COLOR = '#CC79A7'
PROJECTIONS = [(0, 1), (1, 3), (2, 0)]
OUT_DIR = ROOT / 'results' / 'synthetic' / 'figures' / 'm4_low_seed103_method_projections'
OUT_DIR.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({
    'font.family': 'DejaVu Serif',
    'font.size': 8.5,
    'axes.titlesize': 10.0,
    'axes.labelsize': 8.5,
    'xtick.labelsize': 7.5,
    'ytick.labelsize': 7.5,
    'legend.fontsize': 8.0,
    'savefig.facecolor': 'white',
    'axes.facecolor': 'white',
})
print('Cenário:', SCENARIO, '| semente:', SEED)

In [ ]:
runs = pd.read_csv(ROOT / 'results' / 'synthetic' / 'tables' / 'full_method_runs.csv')
metrics = pd.read_csv(ROOT / 'results' / 'synthetic' / 'tables' / 'full_metrics.csv')
selected = runs.loc[
    runs['scenario'].eq(SCENARIO)
    & runs['seed'].eq(SEED)
    & runs['method'].isin(METHOD_ORDER)
].copy()
assert set(selected['method']) == set(METHOD_ORDER)

scenario_data = np.load(ROOT / 'data' / 'generated' / f'{SCENARIO}_scenario.npz', allow_pickle=False)
anchors = np.asarray(scenario_data['anchors'], dtype=float)
ideal = np.asarray(scenario_data['ideal_true'], dtype=float)
nadir = np.asarray(scenario_data['nadir_true'], dtype=float)
scenario_data.close()
scale = nadir - ideal
assert anchors.shape == (4, 3) and np.all(scale > 0)

def objective_values(x):
    return np.sum((x[:, None, :] - anchors[None, :, :]) ** 2, axis=2)

def normalize(values):
    normalized = (values - ideal) / scale
    return np.clip(normalized, 0.0, 1.0), normalized

fronts = {}
audit_rows = []
complete_metrics = metrics.loc[
    metrics['scenario'].eq(SCENARIO)
    & metrics['seed'].eq(SEED)
    & metrics['comparison'].eq('complete')
]
for method in METHOD_ORDER:
    record = selected.loc[selected['method'].eq(method)].iloc[0]
    checkpoint = ROOT / record['checkpoint']
    data = np.load(checkpoint, allow_pickle=False)
    x = np.asarray(data['X'], dtype=float)
    success = np.asarray(data['success'], dtype=bool)
    data.close()
    feasible = np.sum(x * x, axis=1) <= ALPHA ** 2 + 1e-8
    f = objective_values(x[success & feasible])
    keep = NonDominatedSorting().do(f, only_non_dominated_front=True)
    f = f[keep]
    expected = int(complete_metrics.loc[complete_metrics['method'].eq(method), 'n'].iloc[0])
    assert len(f) == expected
    f_plot, f_unclipped = normalize(f)
    fronts[method] = f_plot
    audit_rows.append({
        'scenario': SCENARIO, 'seed': SEED, 'method': METHOD_LABEL[method],
        'n_non_dominated': len(f),
        'values_below_zero_before_clipping': int((f_unclipped < 0).sum()),
        'values_above_one_before_clipping': int((f_unclipped > 1).sum()),
        'checkpoint': record['checkpoint'],
    })

anchor_objectives = objective_values(anchors)
anchor_plot, anchor_unclipped = normalize(anchor_objectives)
assert np.allclose(anchor_plot, anchor_unclipped)
audit = pd.DataFrame(audit_rows)
audit

In [ ]:
figure, axes = plt.subplots(
    len(METHOD_ORDER), len(PROJECTIONS),
    figsize=(16.0 / 2.54, 23.5 / 2.54),
    sharex=True, sharey=True,
)
figure.subplots_adjust(left=0.11, right=0.99, top=0.975, bottom=0.105, hspace=0.18, wspace=0.24)

for row, method in enumerate(METHOD_ORDER):
    values = fronts[method]
    for column, (i, j) in enumerate(PROJECTIONS):
        axis = axes[row, column]
        axis.scatter(
            values[:, i], values[:, j],
            s=14, c=METHOD_COLOR[method], alpha=0.72,
            edgecolors='white', linewidths=0.25, rasterized=True, zorder=2,
        )
        axis.scatter(
            anchor_plot[:, i], anchor_plot[:, j],
            marker='*', s=80, c=STAR_COLOR, edgecolors='0.20',
            linewidths=0.55, zorder=4, clip_on=False,
        )
        axis.set_xlim(-0.025, 1.025)
        axis.set_ylim(-0.025, 1.025)
        axis.set_aspect('equal', adjustable='box')
        axis.set_xticks([0.0, 0.25, 0.5, 0.75, 1.0])
        axis.set_yticks([0.0, 0.25, 0.5, 0.75, 1.0])
        axis.grid(color='0.82', linewidth=0.45, alpha=0.55)
        axis.set_axisbelow(True)
        for spine in axis.spines.values():
            spine.set_color('0.35')
            spine.set_linewidth(0.65)
        if row == len(METHOD_ORDER) - 1:
            axis.set_xlabel(rf'$f_{{{i + 1}}}$ normalizada')
        axis.set_ylabel(rf'$f_{{{j + 1}}}$ normalizada')
    axes[row, 0].set_title(
        METHOD_LABEL[method], loc='left', pad=3, fontsize=9.0,
        color=METHOD_COLOR[method], fontweight='bold',
    )

axes[0, 0].set_title(r'$f_1 \times f_2$')
axes[0, 1].set_title(r'$f_2 \times f_4$')
axes[0, 2].set_title(r'$f_3 \times f_1$')
star_handle = Line2D(
    [0], [0], marker='*', linestyle='none', markerfacecolor=STAR_COLOR,
    markeredgecolor='0.20', markeredgewidth=0.55, markersize=9,
    label='ótimos individuais verdadeiros',
)
figure.legend(handles=[star_handle], loc='lower center', bbox_to_anchor=(0.57, 0.018), frameon=False)

png_path = OUT_DIR / 'fig_m4_low_seed103_projecoes_f1f2_f2f4_f3f1.png'
pdf_path = OUT_DIR / 'fig_m4_low_seed103_projecoes_f1f2_f2f4_f3f1.pdf'
figure.savefig(png_path, dpi=300, bbox_inches='tight', pad_inches=0.015)
figure.savefig(pdf_path, bbox_inches='tight', pad_inches=0.015)
plt.close(figure)
print('Figura:', png_path.relative_to(ROOT))

In [ ]:
audit_path = OUT_DIR / 'm4_low_seed103_projection_audit.csv'
audit.to_csv(audit_path, index=False)
metadata = {
    'scenario': SCENARIO,
    'seed': SEED,
    'comparison': 'complete front',
    'methods': [METHOD_LABEL[method] for method in METHOD_ORDER],
    'projections': [['f1', 'f2'], ['f2', 'f4'], ['f3', 'f1']],
    'normalization': 'true ideal-nadir per objective, clipped to [0, 1] for plotting',
    'ideal': ideal.tolist(),
    'nadir': nadir.tolist(),
    'optimal_points': 'objective values at the four scenario anchors',
    'optimal_point_marker': 'pink star',
    'method_colors': {METHOD_LABEL[key]: value for key, value in METHOD_COLOR.items()},
}
metadata_path = OUT_DIR / 'm4_low_seed103_projection_metadata.json'
metadata_path.write_text(json.dumps(metadata, indent=2, ensure_ascii=False), encoding='utf-8')
for artifact in [png_path, pdf_path, audit_path, metadata_path]:
    assert artifact.exists() and artifact.stat().st_size > 0
print(audit[['method', 'n_non_dominated', 'values_above_one_before_clipping']].to_string(index=False))